In [ ]:
# =============================================================================
# Catalog slug (shared): maps a login to the participant catalog retailhub_<slug>.
# Run with %run ./catalog_slug from 00_pre_config (trainer, creates the catalogs) and 00_setup
# (participant, finds the catalog). The Workshop Guide app (apps/workshop-guide/server/slug.ts)
# uses the same rule; slug-parity.test.ts runs the code of this notebook and compares both.
# =============================================================================
import re

CATALOG_PREFIX = "retailhub"
# Workshop Guide progress table and quiz key (00_pre_config Step 6); reserved, never a participant catalog
PROGRESS_CATALOG = f"{CATALOG_PREFIX}_dea_trainer"

# Extra trainer accounts: exact login part before "@" (lowercase) -> "trainer". Edit per delivery; do not commit
# real logins. Same logins go to the app secret trainer-accounts. Logins trainer, trainer1, trener02, ... map automatically.
TRAINER_OVERRIDES = {}  # e.g. {"jan.kowalski": "trainer"}
assert all(v == "trainer" for v in TRAINER_OVERRIDES.values()), "the app maps trainer accounts to 'trainer' only"


def catalog_slug(login):
    """jan.kowalski@x.pl -> jan_kowalski, trener02@x.pl -> trainer, 12345@x.pl -> user"""
    prefix = login.split("@")[0]
    if prefix.lower() in TRAINER_OVERRIDES:
        return TRAINER_OVERRIDES[prefix.lower()]
    if re.fullmatch(r"(trainer|trener)\d*", prefix.lower()):
        return "trainer"
    slug = re.sub(r"[^a-zA-Z0-9]", "_", prefix).lower()
    slug = re.sub(r"_+", "_", slug)
    slug = re.sub(r"^[0-9_]+", "", slug)  # a catalog name must not start with a digit
    slug = slug.strip("_") or "user"
    # dea.trainer@... must not land in the progress catalog
    return f"user_{slug}" if f"{CATALOG_PREFIX}_{slug}" == PROGRESS_CATALOG else slug


# Self-check (runs on every %run, costs nothing)
assert catalog_slug("jan.kowalski@a.pl") == "jan_kowalski"
assert catalog_slug("jan.kowalski@b.com") == "jan_kowalski"  # same slug: 00_pre_config stops on this collision
assert catalog_slug("Jan.Kowalski+dbx@X.pl") == "jan_kowalski_dbx"
assert catalog_slug("123@x.com") == "user"
assert catalog_slug("1jan@x.pl") == "jan"
assert catalog_slug("trener02@x.pl") == catalog_slug("Trainer@x.pl") == "trainer"
assert catalog_slug("trainer.anna@x.pl") == "trainer_anna"  # full match only, no substring hits
assert catalog_slug("trener02.kowalski@x.pl") == "trener02_kowalski"
assert catalog_slug("dea.trainer@x.pl") == "user_dea_trainer"  # reserved PROGRESS_CATALOG